# Airline Review Analysis - Notebook 1: Data Cleaning

In this notebook I load the Skytrax airline reviews dataset, look at what is in it, and clean it so it can be used for the NLP tasks in the next notebooks.

The original Excel file is only read, never changed. The cleaned data is saved as a separate CSV file.

Related syllabus topic: Unit 1 (challenges of NLP). Real text data is messy, so cleaning is the first stage of any NLP system.

## 1. Loading the data

In [1]:
import pandas as pd

df_raw = pd.read_excel("Skytrax_Airline_Review_Data.xlsx")
print(df_raw.shape)
df_raw.head()

(64740, 19)


,review_date,over_all_rating,review_header,trip_verified,review_text,aircraft,type_of_traveller,seat_type,route,date_flown,seat_comfort,cabin_staff_service,food__beverages,inflight_entertainment,ground_service,wifi__connectivity,value_for_money,recommended,airline_name
0,2025-08-04,1.0,"""showed zero empathy for passengers""",Trip Verified,My afternoon flight was delayed and eventuall...,NaN,business,economy_class,dubai_to_kochi,august_2025,1.0,2.0,2.0,2.0,1.0,1.0,1.0,no,spicejet
1,2025-07-21,2.0,"""Extremely dissatisfied""",Trip Verified,The flight was postponed 3 times with insuffi...,NaN,family_leisure,economy_class,pune_to_dubai,july_2025,1.0,3.0,NaN,NaN,3.0,NaN,2.0,no,spicejet
2,2025-04-19,2.0,"""Very bad experience""",Trip Verified,They will charge you for your hand baggage if...,NaN,business,economy_class,mumbai_to_kolkata,april_2025,3.0,3.0,3.0,NaN,1.0,NaN,2.0,no,spicejet
3,2025-03-12,1.0,“What a difficult airline”,Trip Verified,The flight was delayed from 15:20 to 16:25. I...,NaN,business,economy_class,"pune,_india_to_new_delhi,_india",march_2025,3.0,4.0,NaN,NaN,5.0,NaN,1.0,no,spicejet
4,2025-03-02,2.0,“Poor chaotic boarding”,Trip Verified,No crowd control and boarding was like cattle...,boeing_747,solo_leisure,economy_class,mumbai_to_new_delhi,march_2025,3.0,1.0,3.0,NaN,1.0,NaN,2.0,no,spicejet


The file has 64,740 reviews and 19 columns. Reading the Excel file takes around a minute.

## 2. Understanding the columns

- `review_text` and `review_header`: the review written by the passenger (this is the text I will analyse)
- `over_all_rating`: overall score from 1 to 10
- `recommended`: whether the passenger recommends the airline (yes / no)
- `seat_comfort`, `cabin_staff_service`, `food__beverages`, `inflight_entertainment`, `ground_service`, `wifi__connectivity`, `value_for_money`: ratings from 1 to 5 for each part of the flight
- `type_of_traveller`, `seat_type`, `route`, `aircraft`, `date_flown`: details about the trip
- `trip_verified`: whether the trip was verified by the website
- `airline_name`, `review_date`: airline and date of the review

In [2]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64740 entries, 0 to 64739
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   review_date             64740 non-null  object 
 1   over_all_rating         64624 non-null  float64
 2   review_header           64740 non-null  object 
 3   trip_verified           53047 non-null  object 
 4   review_text             64593 non-null  object 
 5   aircraft                27973 non-null  object 
 6   type_of_traveller       62102 non-null  object 
 7   seat_type               64702 non-null  object 
 8   route                   61879 non-null  object 
 9   date_flown              62004 non-null  object 
 10  seat_comfort            61265 non-null  float64
 11  cabin_staff_service     61038 non-null  float64
 12  food__beverages         52149 non-null  float64
 13  inflight_entertainment  43578 non-null  float64
 14  ground_service          60053 non-null

## 3. Checking for problems

### 3.1 Missing values

In [3]:
missing = df_raw.isna().mean() * 100
missing.round(1).sort_values(ascending=False)

wifi__connectivity        63.5
aircraft                  56.8
inflight_entertainment    32.7
food__beverages           19.4
trip_verified             18.1
ground_service             7.2
cabin_staff_service        5.7
seat_comfort               5.4
route                      4.4
date_flown                 4.2
type_of_traveller          4.1
over_all_rating            0.2
review_text                0.2
seat_type                  0.1
review_date                0.0
review_header              0.0
value_for_money            0.0
recommended                0.0
airline_name               0.0
dtype: float64

The extra ratings like WiFi (63%) and entertainment (33%) are often empty because passengers only rate what they used. I will leave these empty instead of filling them with made-up values. The columns I need most, `review_text` and `recommended`, are almost complete.

### 3.2 Empty, short and duplicate reviews

In [4]:
word_count = df_raw["review_text"].fillna("").str.split().str.len()

print("reviews with no text:", df_raw["review_text"].isna().sum())
print("reviews with less than 10 words:", (word_count < 10).sum())
print("same airline and same text:", df_raw.duplicated(subset=["airline_name", "review_text"]).sum())

reviews with no text: 147
reviews with less than 10 words: 326
same airline and same text: 177


### 3.3 The `trip_verified` column

In [5]:
df_raw["trip_verified"].value_counts(dropna=False).head(10)

trip_verified
Trip Verified              32948
NaN                        11693
Not Verified               10066
Verified Review             9875
Unverified                   120
NotVerified                    5
TripVerified                   4
British Airways Ranking        4
China Southern Ranking         2
Thai Airways Ranking           2
Name: count, dtype: int64

The same meaning is written in several ways ("Trip Verified", "Verified Review", "TripVerified" and so on), and a few rows contain unrelated text. I will group these into three values: verified, unverified and unknown.

### 3.4 The `recommended` column

In [6]:
df_raw["recommended"].value_counts(dropna=False)

recommended
no     39944
yes    24795
NaN        1
Name: count, dtype: int64

## 4. Cleaning

I work on a copy of the data so the raw dataframe stays as it was.

In [7]:
df = df_raw.copy()
print("rows at start:", len(df))

rows at start: 64740


### 4.1 Cleaning the review text

Removing extra spaces from the text. The words themselves are not changed.

In [8]:
df["review_text"] = df["review_text"].fillna("").astype(str)
df["review_text"] = df["review_text"].str.replace(r"\s+", " ", regex=True).str.strip()

df["review_header"] = df["review_header"].fillna("").astype(str)
df["review_header"] = df["review_header"].str.strip().str.strip('"“”')

# number of words in each review
df["n_words"] = df["review_text"].str.split().str.len()

### 4.2 Removing rows that cannot be used

In [9]:
# no text
df = df[df["review_text"] != ""]
print("after removing empty reviews:", len(df))

# too short to say anything useful
df = df[df["n_words"] >= 10]
print("after removing reviews under 10 words:", len(df))

# no clear yes/no label
df = df[df["recommended"].isin(["yes", "no"])]
print("after removing rows without a label:", len(df))

# duplicates
df = df.drop_duplicates(subset=["airline_name", "review_text"])
print("after removing duplicates:", len(df))

after removing empty reviews: 64593
after removing reviews under 10 words: 64414
after removing rows without a label: 64413
after removing duplicates: 64331


### 4.3 Grouping the `trip_verified` values

In [10]:
def clean_verified(value):
    value = str(value).lower()
    if "not verified" in value or "notverified" in value or "unverified" in value:
        return "unverified"
    elif "verified" in value:
        return "verified"
    else:
        return "unknown"

df["verified"] = df["trip_verified"].apply(clean_verified)
df["verified"].value_counts()

verified
verified      42641
unknown       11622
unverified    10068
Name: count, dtype: int64

Most of the `unknown` rows are old reviews from 2015-2016, before the website started adding this tag.

### 4.4 Dates and the yes/no label

In [11]:
df["review_date"] = pd.to_datetime(df["review_date"])
df["year"] = df["review_date"].dt.year

# date_flown is written like 'august_2025'
df["date_flown"] = pd.to_datetime(df["date_flown"].astype(str).str.replace("_", " "), format="%B %Y", errors="coerce")

# yes = 1, no = 0 so that models can use it
df["recommended_bin"] = (df["recommended"] == "yes").astype(int)

### 4.5 Making names easier to read

In [12]:
# 'american-airlines' becomes 'American Airlines'
df["airline"] = df["airline_name"].str.replace("-", " ").str.title()

# 'solo_leisure' becomes 'Solo Leisure'
df["type_of_traveller"] = df["type_of_traveller"].str.replace("_", " ").str.title()
df["seat_type"] = df["seat_type"].str.replace("_", " ").str.title()

df[["airline", "type_of_traveller", "seat_type"]].head()

,airline,type_of_traveller,seat_type
0,Spicejet,Business,Economy Class
1,Spicejet,Family Leisure,Economy Class
2,Spicejet,Business,Economy Class
3,Spicejet,Business,Economy Class
4,Spicejet,Solo Leisure,Economy Class


## 5. Final check

In [13]:
columns = ["airline", "airline_name", "review_date", "year", "date_flown",
           "over_all_rating", "recommended", "recommended_bin",
           "review_header", "review_text", "n_words", "verified",
           "type_of_traveller", "seat_type", "aircraft", "route",
           "seat_comfort", "cabin_staff_service", "food__beverages",
           "inflight_entertainment", "ground_service", "wifi__connectivity", "value_for_money"]

df_clean = df[columns].reset_index(drop=True)

print("reviews before cleaning:", len(df_raw))
print("reviews after cleaning:", len(df_clean))
print("airlines:", df_clean["airline"].nunique())
print("from", df_clean["review_date"].min().date(), "to", df_clean["review_date"].max().date())
print("percent recommended:", round(df_clean["recommended_bin"].mean() * 100, 1))

reviews before cleaning: 64740
reviews after cleaning: 64331
airlines: 109
from 2015-01-05 to 2025-08-07
percent recommended: 38.3


Reading a few reviews to make sure the text looks fine:

In [14]:
for i, row in df_clean.sample(2, random_state=7).iterrows():
    print(row["airline"], "| rating:", row["over_all_rating"], "| recommended:", row["recommended"])
    print(row["review_text"][:500])
    print()

Delta Air Lines | rating: 1.0 | recommended: no
Delta used to be one of the best airlines apart from Alaska, they used to take customer service seriously. I haven’t flown much in the last several years but due to work I am beginning to travel on a weekly basis. Delta has obviously gone downhill in the last several years. Not putting families together when flying, not able to assist with seat changes to accommodate families, and this entire process change. This will be my very last flight with this company. It’s sad how they’ve taken Northwest

Lufthansa | rating: 1.0 | recommended: no
Till now I haven’t received my baggage and now Lufthansa has deleted my reference number from there records. They have claimed through email that my baggage has been tracked by till date none of Lufthansa team is responding to emails and calls. Very bad service waste of money … as well as big loss to me.



## 6. Saving the cleaned data

In [15]:
import os
os.makedirs("data", exist_ok=True)

df_clean.to_csv("data/reviews_clean.csv", index=False)
print("saved", len(df_clean), "reviews to data/reviews_clean.csv")

saved 64331 reviews to data/reviews_clean.csv


## Summary

- The raw dataset has 64,740 reviews from 109 airlines (2015 to 2025).
- After removing empty, very short, unlabelled and duplicate reviews, 64,331 reviews are left.
- Missing sub-ratings were left empty.
- Inconsistent `trip_verified` values were grouped into three categories.
- The cleaned data is saved in `data/reviews_clean.csv` and will be used in the next notebooks.

Next notebook: exploratory data analysis (EDA) with charts.